In [1]:
# Task 2 - Exploratory Data Analysis

import pandas as pd

df = pd.read_csv("WA_Fn-UseC_-HR-Employee-Attrition.csv")

print("Dataset Shape:", df.shape)
print("\nDataset Preview:")
display(df.head())

Dataset Shape: (1470, 35)

Dataset Preview:


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [2]:
# 1. Summary Statistics
numeric_columns = [
    "Age", "MonthlyIncome",
    "YearsAtCompany", "TotalWorkingYears"
]

display(df[numeric_columns].agg(
    ["mean", "median", "min", "max", "std"]
).round(2))


# 2. Department-wise GroupBy Analysis
department_analysis = df.groupby("Department").agg(
    Employee_Count=("EmployeeNumber", "count"),
    Average_Age=("Age", "mean"),
    Average_Salary=("MonthlyIncome", "mean"),
    Average_Experience=("TotalWorkingYears", "mean")
)

display(department_analysis.round(2))


# 3. Top 3 Correlations
correlation = df.corr(numeric_only=True)

correlation_pairs = correlation.where(
    ~pd.DataFrame(
        [[i >= j for j in range(len(correlation.columns))]
         for i in range(len(correlation.columns))],
        index=correlation.index,
        columns=correlation.columns
    )
).stack().reset_index()

correlation_pairs.columns = [
    "Column 1", "Column 2", "Correlation"
]

top_3 = correlation_pairs.reindex(
    correlation_pairs["Correlation"].abs().sort_values(
        ascending=False
    ).index
).head(3)

display(top_3)


# 4. Business Questions

attrition_rate = (df["Attrition"] == "Yes").mean() * 100
print(f"Q1. Overall attrition rate: {attrition_rate:.2f}%")

salary_by_department = df.groupby(
    "Department"
)["MonthlyIncome"].mean()

print(
    f"Q2. Highest average salary department: "
    f"{salary_by_department.idxmax()} "
    f"(Rs. {salary_by_department.max():.2f})"
)

overtime_attrition = pd.crosstab(
    df["OverTime"],
    df["Attrition"],
    normalize="index"
) * 100

print("\nQ3. Attrition percentage based on overtime:")
display(overtime_attrition.round(2))


# 5. Merge with Department Budget
department_budget = pd.DataFrame({
    "Department": [
        "Sales",
        "Research & Development",
        "Human Resources"
    ],
    "Annual_Budget": [5000000, 8000000, 2000000]
})

merged_data = pd.merge(
    department_analysis.reset_index(),
    department_budget,
    on="Department",
    how="left"
)

display(merged_data.round(2))


# 6. Bonus Pivot Table
pivot = pd.pivot_table(
    df,
    values="MonthlyIncome",
    index="Department",
    columns="Attrition",
    aggfunc="mean"
)

display(pivot.round(2))

,Age,MonthlyIncome,YearsAtCompany,TotalWorkingYears
mean,36.92,6502.93,7.01,11.28
median,36.00,4919.00,5.00,10.00
min,18.00,1009.00,0.00,0.00
max,60.00,19999.00,40.00,40.00
std,9.14,4707.96,6.13,7.78


,Employee_Count,Average_Age,Average_Salary,Average_Experience
Department,,,,
Human Resources,63,37.81,6654.51,11.56
Research & Development,961,37.04,6281.25,11.34
Sales,446,36.54,6959.17,11.11


,Column 1,Column 2,Correlation
245,JobLevel,MonthlyIncome,0.950300
253,JobLevel,TotalWorkingYears,0.782208
379,PercentSalaryHike,PerformanceRating,0.773550


Q1. Overall attrition rate: 16.12%
Q2. Highest average salary department: Sales (Rs. 6959.17)

Q3. Attrition percentage based on overtime:


Attrition,No,Yes
OverTime,,
No,89.56,10.44
Yes,69.47,30.53


,Department,Employee_Count,Average_Age,Average_Salary,Average_Experience,Annual_Budget
0,Human Resources,63,37.81,6654.51,11.56,2000000
1,Research & Development,961,37.04,6281.25,11.34,8000000
2,Sales,446,36.54,6959.17,11.11,5000000


Attrition,No,Yes
Department,,
Human Resources,7345.98,3715.75
Research & Development,6630.33,4108.08
Sales,7232.24,5908.46
